# ResNet-50 Visual Product-Group Classification

This notebook documents an optional second-stage classification experiment for the retail shelf project.

Important notes:

- The original detection dataset was not manually labeled into true SKU classes.
- The 91 classes used here are visual product-group pseudo-labels generated from product crops.
- The visual groups were produced through DINOv2 visual embeddings and Mini-Batch K-Means clustering.
- This experiment should be interpreted as visual product-group classification, not verified commercial SKU recognition.
- Dataset export/archive cells were removed from this public version.


In [ ]:
import os
import cv2
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from torchvision import transforms

# Input dataset directory
DATA_DIR = "/kaggle/input/label-cluster1"

# Output cleaned dataset directory
OUTPUT_DIR = "/kaggle/working/cleaned_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Preprocessing parameters
TARGET_SIZE = 224
BLUR_THRESHOLD = 60          # Below this → considered blurry
MIN_ASPECT_RATIO = 0.30      # Wrong crop filter
MAX_ASPECT_RATIO = 3.50


In [ ]:
import os
import cv2
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image
from tqdm import tqdm
from torchvision import transforms

DATA_DIR = "/kaggle/input/label-cluster1"
OUTPUT_DIR = "/kaggle/working/cleaned_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_SIZE = 224
BLUR_THRESHOLD = 60     # قيمة معيارية — أقل منها يعتبر Blurry
MIN_ASPECT = 0.3        # crop خاطئ
MAX_ASPECT = 3.5        # crop خاطئ


In [ ]:
label_counts = {}
labels = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]

for label in labels:
    img_files = [
        f for f in os.listdir(os.path.join(DATA_DIR, label))
        if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))
    ]
    label_counts[label] = len(img_files)

print("Number of labels:", len(label_counts))
for label, count in label_counts.items():
    print(f"{label}: {count}")


In [ ]:
plt.figure(figsize=(18, 6))
plt.bar(label_counts.keys(), label_counts.values())
plt.xticks(rotation=90)
plt.title("Class Distribution")
plt.show()


In [ ]:
STRONG_CLASSES = [cls for cls, cnt in label_counts.items() if cnt > 5000]
WEAK_CLASSES   = [cls for cls, cnt in label_counts.items() if cnt < 1000]

print("\nStrong Classes (>5000 images):")
print(STRONG_CLASSES)

print("\nWeak Classes (<1000 images):")
print(WEAK_CLASSES)


In [ ]:
def show_cluster_samples(label, n=20):
    class_dir = os.path.join(DATA_DIR, label)
    images = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg','.png','.jpeg'))]

    sample = random.sample(images, min(n, len(images)))

    plt.figure(figsize=(15, 8))
    for i, img_name in enumerate(sample):
        img_path = os.path.join(class_dir, img_name)
        img = Image.open(img_path)

        plt.subplot(4, 5, i+1)
        plt.imshow(img)
        plt.axis("off")

    plt.suptitle(f"Cluster Visual Review: {label}")
    plt.show()

# Example:
show_cluster_samples("Beechams_Cold_Flu")


In [ ]:
#5) Data Cleaning
import os
import cv2
import shutil
from tqdm import tqdm

EXT = {'.jpg', '.jpeg', '.png'}  # faster lookup
RESIZE_FOR_BLUR_CHECK = (128, 128)  # downscale for Laplacian speedup

def fast_is_blurry(path, threshold=60):
    """Fast blur detection using grayscale + small resized image."""
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return True

    # Downscale image before Laplacian → MUCH faster
    small = cv2.resize(img, RESIZE_FOR_BLUR_CHECK, interpolation=cv2.INTER_AREA)
    fm = cv2.Laplacian(small, cv2.CV_64F).var()
    return fm < threshold


def fast_wrong_crop(path, min_ar=0.3, max_ar=3.5):
    """Fast aspect ratio check without heavy computations."""
    img = cv2.imread(path)
    if img is None:
        return True
    h, w = img.shape[:2]
    ar = h / w
    return ar < min_ar or ar > max_ar

In [ ]:
for label in tqdm(labels, desc="Fast Cleaning Dataset"):
    src_dir = os.path.join(DATA_DIR, label)
    dst_dir = os.path.join(OUTPUT_DIR, label)
    os.makedirs(dst_dir, exist_ok=True)

    for fname in os.listdir(src_dir):
        if not any(fname.lower().endswith(ext) for ext in EXT):
            continue

        src_path = os.path.join(src_dir, fname)

        # Blur check
        if fast_is_blurry(src_path):
            continue

        # Wrong crop check
        if fast_wrong_crop(src_path):
            continue

        # Copy clean image
        shutil.copy2(src_path, os.path.join(dst_dir, fname))


In [ ]:
clean_label_counts = {}
clean_labels = [d for d in os.listdir(OUTPUT_DIR) if os.path.isdir(os.path.join(OUTPUT_DIR, d))]

for label in clean_labels:
    img_files = [
        f for f in os.listdir(os.path.join(OUTPUT_DIR, label))
        if f.lower().endswith(('.jpg','.jpeg','.png'))
    ]
    clean_label_counts[label] = len(img_files)

print("Cleaned counts:")
for k, v in clean_label_counts.items():
    print(k, v)


In [ ]:
#6) Standardization (Resize + Normalization)
standard_transform = transforms.Compose([
    transforms.Resize((TARGET_SIZE, TARGET_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
# 7) Augmentation for Weak Classes

aug_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4),
    transforms.RandomPerspective(distortion_scale=0.5, p=0.4)
])

AUG_TARGET = 2000  # Minimum images per weak class

for label in WEAK_CLASSES:
    class_dir = os.path.join(OUTPUT_DIR, label)
    imgs = [f for f in os.listdir(class_dir) if f.lower().endswith('.jpg')]

    print(f"Augmenting class: {label}, current: {len(imgs)} → target: {AUG_TARGET}")

    while len(imgs) < AUG_TARGET:
        img_name = random.choice(imgs)
        img_path = os.path.join(class_dir, img_name)

        img = Image.open(img_path)
        aug = aug_transform(img)

        new_name = f"aug_{random.randint(0, 999999)}.jpg"
        aug.save(os.path.join(class_dir, new_name))

        imgs.append(new_name)


In [ ]:
#🚀 Train/Val/Test Split Code (70/15/15)
import os
import shutil
import random
from tqdm import tqdm

SOURCE_DIR = "/kaggle/working/cleaned_dataset"
SPLIT_DIR = "/kaggle/working/dataset_split"

train_ratio = 0.70
val_ratio = 0.15
test_ratio = 0.15

random.seed(42)

# Create split folders
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(SPLIT_DIR, split), exist_ok=True)

labels = [d for d in os.listdir(SOURCE_DIR) if os.path.isdir(os.path.join(SOURCE_DIR, d))]

for label in tqdm(labels, desc="Splitting Dataset"):
    class_dir = os.path.join(SOURCE_DIR, label)

    images = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    random.shuffle(images)

    n = len(images)
    train_end = int(n * train_ratio)
    val_end = train_end + int(n * val_ratio)

    train_imgs = images[:train_end]
    val_imgs = images[train_end:val_end]
    test_imgs = images[val_end:]

    # Create class folders inside split directories
    os.makedirs(os.path.join(SPLIT_DIR, "train", label), exist_ok=True)
    os.makedirs(os.path.join(SPLIT_DIR, "val", label), exist_ok=True)
    os.makedirs(os.path.join(SPLIT_DIR, "test", label), exist_ok=True)

    # Copy images
    for img in train_imgs:
        shutil.copy2(os.path.join(class_dir, img), os.path.join(SPLIT_DIR, "train", label, img))

    for img in val_imgs:
        shutil.copy2(os.path.join(class_dir, img), os.path.join(SPLIT_DIR, "val", label, img))

    for img in test_imgs:
        shutil.copy2(os.path.join(class_dir, img), os.path.join(SPLIT_DIR, "test", label, img))

print("\nDataset split completed!")


In [ ]:
import os
import random
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights

from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

# =====================
# Basic Config
# =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

DATA_ROOT = "/kaggle/working/dataset_split"
BATCH_SIZE = 64
NUM_WORKERS = 0
NUM_EPOCHS = 10
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# ImageNet normalization
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_dataset = datasets.ImageFolder(os.path.join(DATA_ROOT, "train"), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_ROOT, "val"),   transform=eval_transform)
test_dataset  = datasets.ImageFolder(os.path.join(DATA_ROOT, "test"),  transform=eval_transform)

num_classes = len(train_dataset.classes)
print("Num classes:", num_classes)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# Class Weights
train_labels = [label for _, label in train_dataset.samples]
class_counts = Counter(train_labels)
num_samples = len(train_dataset)

class_weights = torch.tensor(
    [num_samples / (num_classes * class_counts[c]) for c in range(num_classes)],
    dtype=torch.float32
).to(device)

print("Class weights:", class_weights)


In [ ]:
weights = ResNet50_Weights.IMAGENET1K_V2
model = resnet50(weights=weights)

# Replace classifier
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.3, patience=2
)

print("Model initialized:", next(model.parameters()).device)


class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.counter = 0
        self.best_acc = 0
        self.early_stop = False

    def step(self, val_acc):
        if val_acc > self.best_acc:
            self.best_acc = val_acc
            self.counter = 0
        else:
            self.counter += 1
        if self.counter >= self.patience:
            self.early_stop = True


early_stopper = EarlyStopping()


In [ ]:
from tqdm.auto import tqdm
from torch.amp import autocast, GradScaler

scaler = GradScaler('cuda')

def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    total, correct, running_loss = 0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total


In [ ]:
best_val_acc = 0
best_path = "/kaggle/working/resnet50_best.pth"

print("\n===== GPU CHECK =====")
print("GPUs:", torch.cuda.device_count())
print("Model device:", next(model.parameters()).device)

for epoch in range(NUM_EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{NUM_EPOCHS} =====")

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)
    val_loss, val_acc = eval_one_epoch(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_path)
        print(f"🔥 Best updated → {best_val_acc*100:.2f}%")

    early_stopper.step(val_acc)
    if early_stopper.early_stop:
        print("⛔ Early stopping triggered")
        break

print("\nTraining Done!")
print("Best Validation Accuracy:", best_val_acc * 100)


In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

num_classes = len(train_dataset.classes)

# Create same model architecture used during training
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

best_model_path = "/kaggle/working/resnet50_best.pth"

# Load state dict
state_dict = torch.load(best_model_path, map_location=device)

# ----------- FIX: remove 'module.' prefix if exists -----------
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith("module."):
        new_state_dict[k[len("module."):]] = v   # remove 'module.'
    else:
        new_state_dict[k] = v

# Load corrected weights
model.load_state_dict(new_state_dict)

model = model.to(device)
model.eval()

print("🔥 Best model loaded successfully (with DP fix)!")


In [ ]:
from tqdm import tqdm
import numpy as np

test_loss = 0
correct = 0
total = 0
criterion = nn.CrossEntropyLoss()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)

        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_loss = test_loss / total
test_acc = correct / total * 100

print(f"\n===== TEST RESULTS =====")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        labels = labels.cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels)

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(14, 10))
sns.heatmap(cm, cmap="Blues", annot=False)
plt.title("Confusion Matrix")
plt.show()


In [ ]:
from sklearn.metrics import classification_report

print("\n===== Classification Report =====")
print(classification_report(all_labels, all_preds,
                            target_names=train_dataset.classes))


In [ ]:
import numpy as np

class_correct = np.zeros(num_classes)
class_total = np.zeros(num_classes)

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(1)

        for label, pred in zip(labels, preds):
            class_total[label] += 1
            if pred == label:
                class_correct[label] += 1

print("===== Per-Class Accuracy =====")
for i, cls_name in enumerate(train_dataset.classes):
    acc = 100 * class_correct[i] / class_total[i]
    print(f"{cls_name}: {acc:.2f}%")


In [ ]:
from sklearn.metrics import classification_report

print("===== Classification Report =====")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))


In [ ]:
top5_correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, top5 = outputs.topk(5, dim=1)

        for i in range(labels.size(0)):
            if labels[i] in top5[i]:
                top5_correct += 1

        total += labels.size(0)

top5_acc = 100 * top5_correct / total
print(f"🥇 Top-5 Accuracy: {top5_acc:.2f}%")


In [ ]:
import random
import matplotlib.pyplot as plt

model.eval()

for _ in range(5):  # Show 5 random examples
    idx = random.randint(0, len(test_dataset)-1)
    img, true_label = test_dataset[idx]
    img_tensor = img.unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img_tensor)
        pred = output.argmax(1).item()

    plt.imshow(test_dataset[idx][0].permute(1, 2, 0) * torch.tensor(std) + torch.tensor(mean))
    plt.title(f"True: {test_dataset.classes[true_label]}\nPred: {test_dataset.classes[pred]}")
    plt.axis("off")
    plt.show()


In [ ]:
import pandas as pd

records = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        labels = labels.cpu().numpy()

        for t, p in zip(labels, preds):
            records.append({
                "true_label": train_dataset.classes[t],
                "predicted_label": train_dataset.classes[p]
            })

df = pd.DataFrame(records)
df.to_csv("/kaggle/working/test_predictions.csv", index=False)

print("📁 Saved test predictions to test_predictions.csv")
